# Grocery Sales Drivers: Analysis

**Question:** What drives grocery sales (seasonality, monthly timing, promotions, holidays), and which product categories are most sensitive to each?

**Data:** Kaggle "Store Sales - Time Series Forecasting" (Favorita), loaded into SQLite in notebook 01. "Sales" are unit counts, not revenue.

**Data quality note:** Produce is only recorded in some months before mid-2015, and several product families first appear in 2014 (see the data quality checks below). So seasonality is analyzed without Produce, and category growth uses 2016-2017 only.

In [1]:
import pandas as pd, sqlite3
con = sqlite3.connect("../grocery.db")

## Q1: Seasonality and monthly timing

The next four cells are my first pass and include Produce. I found the Produce data was incomplete, so the final Q1 numbers come from the reruns without Produce further down. Weekday numbering in the next cell is 0 = Sunday.

In [2]:
pd.read_sql("""
SELECT strftime('%w', date) AS dow, ROUND(AVG(total_sales)) AS avg_daily_sales
FROM (SELECT date, SUM(sales) AS total_sales FROM train GROUP BY date)
GROUP BY dow ORDER BY dow
""", con)

,dow,avg_daily_sales
0,0,825218.0
1,1,617543.0
2,2,569926.0
3,3,593245.0
4,4,505269.0
5,5,579574.0
6,6,772206.0


In [3]:
daily = pd.read_sql("SELECT date, SUM(sales) AS total_sales FROM train GROUP BY date", con)
daily["date"] = pd.to_datetime(daily["date"])
daily["yr"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month

full = daily[daily["yr"] < 2017].copy()
full["rel"] = full["total_sales"] / full.groupby("yr")["total_sales"].transform("mean")
full.groupby("month")["rel"].mean().round(3)


month
1     0.907
2     0.836
3     0.950
4     0.886
5     0.899
6     0.952
7     1.017
8     0.949
9     1.074
10    1.068
11    1.111
12    1.349
Name: rel, dtype: float64

In [4]:
d = daily[daily["date"] < "2017-08-01"].copy()   # drop the partial last month
d["dow"] = d["date"].dt.day_of_week
d["dom"] = d["date"].dt.day
d["rel"] = d["total_sales"] / d.groupby(["yr", "month", "dow"])["total_sales"].transform("mean")
d.groupby("dom")["rel"].mean().round(3)

dom
1     1.091
2     1.138
3     1.112
4     1.102
5     1.087
6     1.063
7     1.050
8     0.994
9     0.945
10    0.935
11    0.956
12    0.967
13    0.968
14    0.967
15    0.974
16    0.983
17    0.980
18    1.000
19    0.994
20    1.021
21    1.016
22    0.980
23    0.958
24    0.951
25    0.938
26    0.951
27    0.948
28    0.967
29    0.958
30    0.974
31    1.038
Name: rel, dtype: float64

In [5]:
pd.read_sql("""
SELECT strftime('%d', date) AS day_of_month, ROUND(AVG(total_sales)) AS avg_daily_sales
FROM (SELECT date, SUM(sales) AS total_sales FROM train GROUP BY date)
GROUP BY day_of_month ORDER BY day_of_month
""", con)

,day_of_month,avg_daily_sales
0,01,712046.0
1,02,743478.0
2,03,716647.0
3,04,699982.0
4,05,683849.0
5,06,664790.0
6,07,648920.0
7,08,627407.0
8,09,623098.0
9,10,603813.0


## Q2: Category mix

Share of unit sales by product family and year. 2013-2015 shares aren't comparable to later years because Produce and several other families are missing or only partly recorded in the early data. The 2016 shares are reliable: Grocery I, Beverages and Produce made up 66.5% of 2016 unit sales.

In [6]:
cat = pd.read_sql("""
SELECT family, strftime('%Y', date) AS yr, SUM(sales) AS total_sales
FROM train GROUP BY family, yr
""", con)
p = cat.pivot(index="family", columns="yr", values="total_sales")

In [7]:
share = (p / p.sum()) * 100
share.round(1).sort_values("2016", ascending=False)

yr,2013,2014,2015,2016,2017
family,,,,,
GROCERY I,41.5,31.3,31.6,29.8,29.4
BEVERAGES,15.5,19.6,20.1,21.4,22.6
PRODUCE,0.1,11.2,10.6,15.3,15.1
CLEANING,12.5,9.3,9.3,7.8,8.0
DAIRY,5.5,6.4,6.2,5.9,5.9
BREAD/BAKERY,5.0,3.9,4.1,3.6,3.5
POULTRY,3.1,3.5,3.2,2.6,2.5
MEATS,4.6,3.0,2.8,2.4,2.3
PERSONAL CARE,2.7,2.3,2.4,2.1,2.1


Not used: growth from 2014 to 2016 is misleading because many families weren't fully recorded in 2014 (Produce is present in only some months, for example). The corrected growth comparison is further down.

In [8]:
((p["2016"] / p["2014"] - 1) * 100).round(1).sort_values(ascending=False)

family
BOOKS                           inf
BABY CARE                     734.6
MAGAZINES                     729.8
HOME AND KITCHEN II           293.6
SCHOOL AND OFFICE SUPPLIES    153.9
PET SUPPLIES                  128.1
CELEBRATION                    95.8
PRODUCE                        88.2
HOME CARE                      87.1
PLAYERS AND ELECTRONICS        84.4
HOME AND KITCHEN I             84.4
BEAUTY                         78.9
LADIESWEAR                     52.1
BEVERAGES                      50.4
LAWN AND GARDEN                43.5
GROCERY I                      31.1
PERSONAL CARE                  28.8
DAIRY                          25.8
BREAD/BAKERY                   24.6
AUTOMOTIVE                     23.9
DELI                           23.5
FROZEN FOODS                   23.1
LIQUOR,WINE,BEER               22.0
CLEANING                       16.1
PREPARED FOODS                 13.7
MEATS                           9.9
EGGS                            9.1
HARDWARE             

In [9]:
  share["2016"].sort_values(ascending=False).head(3).sum().round(1)

np.float64(66.5)

## Data quality checks

- Produce has tiny volumes in months where only about 24 stores report it and millions of units in months where 47-53 stores do. It's effectively absent in 2013 and patchy until June 2015.
- Nine product families first appear in January 2014.
- The number of stores with sales grows from 47 (2013) to 54 (2017).

This is why seasonality excludes Produce and category growth uses only 2016-2017.

In [10]:
# 1. When does each family first have sales?
first = pd.read_sql("""
SELECT family, MIN(date) AS first_sale_date
FROM train WHERE sales > 0
GROUP BY family
""", con)
first.sort_values("first_sale_date", ascending=False)

,family,first_sale_date
4,BOOKS,2016-10-08
1,BABY CARE,2014-03-01
31,SCHOOL AND OFFICE SUPPLIES,2014-01-02
19,LADIESWEAR,2014-01-02
16,HOME AND KITCHEN II,2014-01-01
27,PLAYERS AND ELECTRONICS,2014-01-01
6,CELEBRATION,2014-01-01
26,PET SUPPLIES,2014-01-01
23,MAGAZINES,2014-01-01
18,HOME CARE,2014-01-01


In [11]:
# 2. How many stores were active each year?
pd.read_sql("""
SELECT strftime('%Y', date) AS yr, COUNT(DISTINCT store_nbr) AS stores_with_sales
FROM train WHERE sales > 0
GROUP BY yr
""", con)

,yr,stores_with_sales
0,2013,47
1,2014,48
2,2015,53
3,2016,53
4,2017,54


In [12]:
# 3. Produce over time: sales and number of stores selling it
pd.read_sql("""
SELECT strftime('%Y-%m', date) AS ym, ROUND(SUM(sales)) AS produce_sales,
       COUNT(DISTINCT CASE WHEN sales > 0 THEN store_nbr END) AS stores_selling
FROM train WHERE family = 'PRODUCE'
GROUP BY ym ORDER BY ym
""", con)

,ym,produce_sales,stores_selling
0,2013-01,0.0,0
1,2013-02,0.0,0
2,2013-03,4037.0,24
3,2013-04,6308.0,24
4,2013-05,8836.0,24
5,2013-06,8631.0,24
6,2013-07,7463.0,24
7,2013-08,8118.0,24
8,2013-09,6821.0,24
9,2013-10,5910.0,24


## Q1 rerun without Produce (final numbers)

Pandas weekday numbering below: 0 = Monday, 6 = Sunday.
- **Weekends:** about 716k units per day vs. about 504k on weekdays, roughly 42% higher. Sunday is about 64% above Thursday.
- **Month** (1.00 = a typical day that year, 2013-2016): December is 1.325, about 32% above a typical day. Other months fall between 0.88 and 1.08. Values below 1.0 are partly mechanical, because December pulls up each year's average.

In [13]:
np_daily = pd.read_sql("""
SELECT date, SUM(sales) AS total_sales
FROM train WHERE family != 'PRODUCE'
GROUP BY date
""", con)
np_daily["date"] = pd.to_datetime(np_daily["date"])
np_daily["yr"] = np_daily["date"].dt.year
np_daily["month"] = np_daily["date"].dt.month

full_np = np_daily[np_daily["yr"] < 2017].copy()
full_np["rel"] = full_np["total_sales"] / full_np.groupby("yr")["total_sales"].transform("mean")
print(full_np.groupby("month")["rel"].mean().round(3))
print(np_daily.groupby(np_daily["date"].dt.day_of_week)["total_sales"].mean().round())

month
1     0.907
2     0.878
3     0.953
4     0.936
5     0.947
6     0.957
7     0.984
8     0.957
9     1.039
10    1.036
11    1.079
12    1.325
Name: rel, dtype: float64
date
0    549712.0
1    500003.0
2    503369.0
3    450306.0
4    517553.0
5    692836.0
6    739128.0
Name: total_sales, dtype: float64


## Q2 rerun: category growth, Jan-Jul 2017 vs. Jan-Jul 2016

The only window where every family is fully recorded in both years. Unit sales for all families grew about 9.9%. Beverages grew fastest among the major families (+16.9%), while Grocery I, the largest at 29.7% of units, grew 8.7%. Caveats: one more store was selling in 2017 (54 vs. 53), and the 2016 base may be inflated by the April 2016 earthquake. The table shows families with at least 1% of units.

In [14]:
yoy = pd.read_sql("""
SELECT family,
       SUM(CASE WHEN strftime('%Y', date) = '2016' THEN sales END) AS s2016,
       SUM(CASE WHEN strftime('%Y', date) = '2017' THEN sales END) AS s2017
FROM train
WHERE strftime('%m', date) BETWEEN '01' AND '07'
  AND strftime('%Y', date) IN ('2016', '2017')
GROUP BY family
""", con)
yoy["share_2016"] = (yoy["s2016"] / yoy["s2016"].sum() * 100).round(1)
yoy["growth_pct"] = ((yoy["s2017"] / yoy["s2016"] - 1) * 100).round(1)
print("all families:", round((yoy["s2017"].sum() / yoy["s2016"].sum() - 1) * 100, 1))
yoy[yoy["share_2016"] >= 1].sort_values("growth_pct", ascending=False)

all families: 9.9


,family,s2016,s2017,share_2016,growth_pct
3,BEVERAGES,3.519973e+07,4.114269e+07,21.3,16.9
7,CLEANING,1.305785e+07,1.452296e+07,7.9,11.2
9,DELI,3.274859e+06,3.641797e+06,2.0,11.2
8,DAIRY,9.752065e+06,1.078116e+07,5.9,10.6
10,EGGS,2.044096e+06,2.259245e+06,1.2,10.5
12,GROCERY I,4.913884e+07,5.343074e+07,29.7,8.7
18,HOME CARE,3.246983e+06,3.530434e+06,2.0,8.7
5,BREAD/BAKERY,5.950197e+06,6.403953e+06,3.6,7.6
30,PRODUCE,2.597675e+07,2.741981e+07,15.7,5.6
24,MEATS,4.019951e+06,4.237932e+06,2.4,5.4


## Day of month without Produce (final)

Each day is compared with other days of the same weekday in the same month. Days 1-7 average 1.097 against 0.969 for days 8-30, so the first week runs about 13% above the rest of the month. There is no visible bump on the 15th. Day 31 is inflated (it only occurs in 7 months and includes Dec 31), so I ignore it. The pattern is consistent with a pay-cycle effect, but the data can't show the cause.

In [15]:
d2 = np_daily[np_daily["date"] < "2017-08-01"].copy()
d2["dow"] = d2["date"].dt.day_of_week
d2["dom"] = d2["date"].dt.day
d2["rel"] = d2["total_sales"] / d2.groupby(["yr", "month", "dow"])["total_sales"].transform("mean")
r = d2.groupby("dom")["rel"].mean()
print(r.round(3))
print("days 1-7:", round(r.loc[1:7].mean(), 3), " days 8-30:", round(r.loc[8:30].mean(), 3))

dom
1     1.099
2     1.146
3     1.118
4     1.107
5     1.092
6     1.066
7     1.052
8     0.993
9     0.942
10    0.934
11    0.954
12    0.965
13    0.965
14    0.965
15    0.972
16    0.981
17    0.979
18    0.999
19    0.994
20    1.022
21    1.017
22    0.977
23    0.956
24    0.947
25    0.935
26    0.948
27    0.946
28    0.966
29    0.955
30    0.973
31    1.039
Name: rel, dtype: float64
days 1-7: 1.097  days 8-30: 0.969
